### LIBRARY IMPORTS

In [1]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
import copy

from sklearn.metrics import accuracy_score

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.cnn_regressor import CNNRegressor

### CONFIGURATION

In [2]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, valid, test = data_manager.load_image_data(
    active_dataset
)

y_test = np.asarray(test.dataset.targets)[test.indices]

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)

y_train, y_valid = processor.transform_target(y_train, y_valid)

test = processor.convert_to_numpy(test) 

### GRADIENT BOOSTING

In [6]:
gb_preds = pd.read_csv(f"{MODELS_PATH}/{active_dataset}/2026_04_07_18_03/predictions.csv")
accuracy_score(y_test, gb_preds[active_dataset_config["target"]])

0.7641

### CONVOLUTIONAL NEURAL NETWORKS

Big CNN total parameters: 122,570

Small CNN total parameters: 7,994

In [3]:
class CNN(CNNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(self, X_train, y_train, X_valid, y_valid):
        
        in_channels = X_train.shape[1]
        output_size = int(np.max(y_train)) + 1

        image_size = X_train.shape[-1]

        conv1_out = image_size - (self.kernel_size - 1)
        pool1_out = conv1_out // self.pool_size

        conv2_out = pool1_out - (self.kernel_size - 1)
        pool2_out = conv2_out // self.pool_size

        conv3_out = pool2_out - (self.kernel_size - 1)
        linear_input = self.channels[1] * conv3_out * conv3_out
        
        self._get_network(in_channels, linear_input, output_size)

        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).long().to(self.device).view(-1)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), self.learning_rate)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                
                loss = criterion(preds, batch_y.long().view(-1))
                loss.backward()
                optimizer.step()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

                pred_labels = val_preds.argmax(dim=1)
                val_acc = (pred_labels == y_valid_t).float().mean().item()

            if (epoch + 1) % 1 == 0:
                print(f"Epoch: {epoch + 1} | Validation Log Loss: {val_loss:.4f} | Validation Accuracy: {val_acc:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= self.patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)

In [5]:
big_cnn = CNN(epochs=100, patience=10, learning_rate=0.001, channels=[32, 64], kernel_size=3, pool_size=2, hidden_size=64, batch_size=256)
big_cnn.fit(X_train, y_train, X_valid, y_valid)

print ("-" * 50)

raw_preds = big_cnn.predict(test) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)

score = accuracy_score(y_test, preds)
print(f"Test Accuracy: {score:.4f}")

Epoch: 1 | Validation Log Loss: 1.6860 | Validation Accuracy: 0.3669
Epoch: 2 | Validation Log Loss: 1.5374 | Validation Accuracy: 0.4408
Epoch: 3 | Validation Log Loss: 1.4231 | Validation Accuracy: 0.4852
Epoch: 4 | Validation Log Loss: 1.4451 | Validation Accuracy: 0.4885
Epoch: 5 | Validation Log Loss: 1.2997 | Validation Accuracy: 0.5361
Epoch: 6 | Validation Log Loss: 1.2544 | Validation Accuracy: 0.5547
Epoch: 7 | Validation Log Loss: 1.1916 | Validation Accuracy: 0.5761
Epoch: 8 | Validation Log Loss: 1.1638 | Validation Accuracy: 0.5876
Epoch: 9 | Validation Log Loss: 1.1469 | Validation Accuracy: 0.6007
Epoch: 10 | Validation Log Loss: 1.1126 | Validation Accuracy: 0.6109
Epoch: 11 | Validation Log Loss: 1.1039 | Validation Accuracy: 0.6146
Epoch: 12 | Validation Log Loss: 1.1094 | Validation Accuracy: 0.6142
Epoch: 13 | Validation Log Loss: 1.0514 | Validation Accuracy: 0.6344
Epoch: 14 | Validation Log Loss: 1.0351 | Validation Accuracy: 0.6360
Epoch: 15 | Validation Log Lo

In [4]:
small_cnn = CNN(epochs=1000, patience=10, learning_rate=0.001, channels=[8, 16], kernel_size=3, pool_size=2, hidden_size=16, batch_size=32)
small_cnn.fit(X_train, y_train, X_valid, y_valid)

print ("-" * 50)

raw_preds = small_cnn.predict(test) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)

score = accuracy_score(y_test, preds)
print(f"Test Accuracy: {score:.4f}")

Epoch: 1 | Validation Log Loss: 1.7970 | Validation Accuracy: 0.3201


KeyboardInterrupt: 